# Gemma 3 on Amazon Bedrock — and why it is nothing like Gemma 4

Gemma 3 and Gemma 4 come from the same provider, one generation apart. Almost
nothing about calling them is the same.

| | Gemma 3 | Gemma 4 |
|---|---|---|
| Endpoints | `bedrock-mantle` **and** `bedrock-runtime` | `bedrock-mantle` only |
| Mantle path prefix | `/v1` | `/openai/v1` |
| Responses API | not supported | supported |
| Token budget | `max_tokens` *or* `max_completion_tokens` | `max_completion_tokens` only |
| `temperature` | free | pinned to the default (1) |
| `top_p` | accepted | rejected |
| Vision | yes | yes |

If you take one thing from this collection, take this table. A generic "how to
call a model on Bedrock" example would be wrong about Gemma 3 or wrong about
Gemma 4 — it cannot be right about both. That is why there is a folder per
family and a notebook per generation.

Everything below is probed live, so you see today's answer rather than the one
that was true when this was written.

**Sizes.** Gemma 3 comes as `4b`, `12b` and `27b`. They share one API surface, so
this notebook uses them interchangeably and points out where size actually
matters.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    api_prefix,
    bands_png,
    converse,
    endpoints_for,
    err,
    post,
    resolve_runtime_id,
)

REGION = "us-east-1"
SMALL = "google.gemma-3-4b-it"
MID = "google.gemma-3-12b-it"
LARGE = "google.gemma-3-27b-it"
GEMMA4 = "google.gemma-4-31b"

print("gemma 3 mantle prefix:", api_prefix(LARGE))
print("gemma 4 mantle prefix:", api_prefix(GEMMA4), " <- different family, different prefix")
print()
for model in (SMALL, MID, LARGE, GEMMA4):
    print(f"{model:<26} {endpoints_for(model)}")


gemma 3 mantle prefix: /v1
gemma 4 mantle prefix: /openai/v1  <- different family, different prefix



google.gemma-3-4b-it       {'mantle': True, 'runtime': True}


google.gemma-3-12b-it      {'mantle': True, 'runtime': True}


google.gemma-3-27b-it      {'mantle': True, 'runtime': True}


google.gemma-4-31b         {'mantle': True, 'runtime': False}


## 1. The divergence, measured

The same five requests against both generations. Read the `detail` column — the
error text is the lesson.


In [2]:
PROBES = [
    ("max_tokens", {"max_tokens": 24}),
    ("max_completion_tokens", {"max_completion_tokens": 24}),
    ("temperature=0.2", {"max_completion_tokens": 24, "temperature": 0.2}),
    ("top_p=0.95", {"max_completion_tokens": 24, "top_p": 0.95}),
]

for model in (LARGE, GEMMA4):
    print(f"{model}   (prefix {api_prefix(model)})")
    for label, extra in PROBES:
        status, data = post(
            f"{api_prefix(model)}/chat/completions",
            {
                "model": model,
                "messages": [{"role": "user", "content": "Reply OK"}],
                **extra,
            },
            region=REGION,
            attempts=1,
        )
        detail = "" if status == 200 else err(data)[:56]
        print(f"    {label:<24} HTTP {status} {detail}")
    print()

print("Gemma 3 accepts every one of them. Gemma 4 rejects three of the four.")


google.gemma-3-27b-it   (prefix /v1)


    max_tokens               HTTP 200 


    max_completion_tokens    HTTP 200 


    temperature=0.2          HTTP 200 


    top_p=0.95               HTTP 200 

google.gemma-4-31b   (prefix /openai/v1)


    max_tokens               HTTP 400 Unsupported parameter: 'max_tokens' is not supported wit


    max_completion_tokens    HTTP 200 


    temperature=0.2          HTTP 400 Unsupported value: 'temperature' does not support 0.2 wi


    top_p=0.95               HTTP 400 Unsupported parameter: 'top_p' is not supported with thi

Gemma 3 accepts every one of them. Gemma 4 rejects three of the four.


## 2. Gemma 3 has no Responses API

Gemma 4 is served on `/openai/v1` and supports the Responses API. Gemma 3 is on
`/v1` and does not. Reaching for the wrong one gives a clear 400, which is the
cheapest way to confirm the shape of a family you have not used before.


In [3]:
status, data = post(
    f"{api_prefix(LARGE)}/responses",
    {"model": LARGE, "input": "Reply OK", "max_output_tokens": 24},
    region=REGION,
    attempts=1,
)
print(f"gemma 3 /responses -> HTTP {status}: {err(data)[:90]}")

status, data = post(
    f"{api_prefix(GEMMA4)}/responses",
    {"model": GEMMA4, "input": "Reply OK", "max_output_tokens": 24},
    region=REGION,
    attempts=1,
)
print(f"gemma 4 /responses -> HTTP {status}")


gemma 3 /responses -> HTTP 400: The model 'google.gemma-3-27b-it' does not support the '/v1/responses' API


gemma 4 /responses -> HTTP 200


## 3. Chat Completions on `bedrock-mantle`

Standard OpenAI shape. Sampling parameters behave the way the OpenAI
documentation says they do, which — as section 1 showed — is *not* true of
Gemma 4.


In [4]:
status, data = post(
    f"{api_prefix(MID)}/chat/completions",
    {
        "model": MID,
        "messages": [
            {"role": "system", "content": "You are terse."},
            {"role": "user", "content": "Why is idempotency useful? One sentence."},
        ],
        "max_tokens": 120,
        "temperature": 0.3,
    },
    region=REGION,
)
if status == 200:
    choice = data["choices"][0]
    print("answer       :", (choice["message"]["content"] or "").strip())
    print("finish_reason:", choice.get("finish_reason"))
    print("usage        :", data.get("usage"))
else:
    print(f"HTTP {status}: {err(data)}")


answer       : Idempotency ensures repeated operations have the same effect as a single operation, simplifying error handling and retries.
finish_reason: stop
usage        : {'completion_tokens': 23, 'prompt_tokens': 23, 'total_tokens': 46}


## 4. The same call on `bedrock-runtime`

Gemma 3 is on both endpoints, so you get a choice Gemma 4 does not offer. Note
the shape change: `content` is a list of blocks, the budget moves into
`inferenceConfig`, and the model ID needs no profile prefix here.


In [5]:
resolved = resolve_runtime_id(LARGE)
print("converse sends:", resolved)

text, response = converse(
    LARGE,
    [
        {
            "role": "user",
            "content": [{"text": "Why is idempotency useful? One sentence."}],
        }
    ],
    system="You are terse.",
    max_tokens=120,
    temperature=0.3,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
if error:
    print("failed:", error[:160])
else:
    print("answer     :", text.strip())
    print("stop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))


converse sends: google.gemma-3-27b-it


answer     : Idempotency ensures an operation can be repeated without changing the result beyond the initial application, simplifying error handling and system reliability.
stop reason: end_turn
tokens     : 50


## 5. Vision, with an image whose answer we already know

Gemma 3 is multimodal. To test that honestly you need an image where you can
check the answer — otherwise a fluent, wrong description looks like success.

`bands_png()` builds a few hundred bytes of PNG from the standard library: solid
horizontal colour bands, in an order we chose. So there is a ground truth, and a
model that answers "red, green, blue" demonstrably looked at the pixels.

The two endpoints want the image differently: Converse takes raw bytes, while the
OpenAI-shaped API wants base64 in a data URL.


In [6]:
import base64

GROUND_TRUTH = ["red", "green", "blue"]
png = bands_png([(220, 30, 30), (30, 140, 60), (40, 70, 200)])
print(f"generated {len(png)} bytes of PNG; bands are {', '.join(GROUND_TRUTH)}")

QUESTION = "List the colours of the horizontal bands, top to bottom. Three words."


def scored(answer: str) -> str:
    """How many of the three known colours did the model actually name?"""
    lowered = (answer or "").lower()
    hits = [colour for colour in GROUND_TRUTH if colour in lowered]
    return f"{len(hits)}/3 correct  {hits}"


# bedrock-runtime: Converse takes the raw bytes.
text, response = converse(
    LARGE,
    [
        {
            "role": "user",
            "content": [
                {"image": {"format": "png", "source": {"bytes": png}}},
                {"text": QUESTION},
            ],
        }
    ],
    max_tokens=60,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
print("\nconverse  :", error[:120] if error else text.strip())
if not error:
    print("            ", scored(text))

# bedrock-mantle: the OpenAI shape wants base64 in a data URL.
data_url = f"data:image/png;base64,{base64.b64encode(png).decode()}"
status, data = post(
    f"{api_prefix(LARGE)}/chat/completions",
    {
        "model": LARGE,
        "max_tokens": 60,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": QUESTION},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
    },
    region=REGION,
)
if status == 200:
    answer = (data["choices"][0]["message"]["content"] or "").strip()
    print("\nmantle    :", answer)
    print("            ", scored(answer))
else:
    print(f"\nmantle    : HTTP {status} {err(data)[:110]}")


generated 308 bytes of PNG; bands are red, green, blue



converse  : Red, green, blue.
             3/3 correct  ['red', 'green', 'blue']



mantle    : Red, green, blue.
             3/3 correct  ['red', 'green', 'blue']


## 6. Does size change the answer?

The three sizes share an API surface, so switching is a one-line change. What
changes is quality and cost. The same vision question across all three is a
cheap way to see where the smallest model stops being sufficient for your task —
which is the only sizing question worth asking.

Watch the token counts and the wording, not just correctness. All three sizes get
the colours right; the two smaller ones ignore "three words" and pad the answer,
so they spend more tokens saying the same thing. Instruction-following is often
what you are buying with size, not raw capability.


In [7]:
print(f"{'model':<26} {'tokens':>7}  answer")
print("-" * 74)
for model in (SMALL, MID, LARGE):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"image": {"format": "png", "source": {"bytes": png}}},
                    {"text": QUESTION},
                ],
            }
        ],
        max_tokens=60,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} {'-':>7}  ERROR {error[:40]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    print(f"{model:<26} {total:>7}  {text.strip()[:44]!r} {scored(text)}")


model                       tokens  answer
--------------------------------------------------------------------------


google.gemma-3-4b-it           305  "Here's the list of colors from the flag, top" 3/3 correct  ['red', 'green', 'blue']


google.gemma-3-12b-it          312  'Certainly! \n\nHere are the colors of the hori' 3/3 correct  ['red', 'green', 'blue']


google.gemma-3-27b-it          290  'Red, green, blue.' 3/3 correct  ['red', 'green', 'blue']


## 7. What to take away

- **Check the family, not the provider.** "Google models on Bedrock" is not a
  useful unit. Gemma 3 and Gemma 4 disagree on the endpoint, the path prefix, the
  token-budget parameter name, and whether `temperature` does anything.
- **Gemma 3 is the more portable of the two.** It is on both endpoints, so you can
  move between Converse and the OpenAI shape without changing model. Gemma 4
  cannot: it is `bedrock-mantle` only.
- **Gemma 4 is the more capable of the two**, and has the Responses API with
  server-side state. If you need that, you accept the single-endpoint dependency.
- **Give vision tests a known answer.** A model that describes an image you cannot
  verify is indistinguishable from a model that guessed.
- **Re-run section 1 before you trust any of this.** Gemma 4's parameter surface
  changed on 12 August 2026 without a release note. Yours may differ from the
  output committed here, and that is the point of keeping the probe in the
  notebook rather than only its result.
